# SeedVR2 Lite — Colab CLI

Runtime: Ubuntu 24.04, Python 3.13, PyTorch 2.11.0+cu128, CUDA toolkit 12.8. Select a GPU runtime before running. This notebook stores weights and results in Google Drive. Edit the Drive root and input filename below.

The setup cell uses main after the CLI change is merged. While reviewing, it automatically checks out the CLI branch if main lacks the runtime check. You can set REPO_REF to another branch or commit.


In [ ]:
import platform
import shutil
import subprocess
import sys
import torch

print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {props.name}, {props.total_memory / 2**30:.1f} GiB")
print("FFmpeg:", shutil.which("ffmpeg") or "not installed")
subprocess.run(["nvidia-smi"], check=False)

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/delvilo/SeedVR2Lite_VideoUpscaler.git"
REPO_REF = "main"
REVIEW_REF = "codex/standalone-colab-cli"
REPO_DIR = Path("/content/SeedVR2Lite_VideoUpscaler")
DRIVE_ROOT = Path("/content/drive/MyDrive/SeedVR2")
INPUT_DIR = DRIVE_ROOT / "input"
MODEL_DIR = DRIVE_ROOT / "models"
OUTPUT_DIR = DRIVE_ROOT / "output"
for directory in (INPUT_DIR, MODEL_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)

if REPO_REF == "main" and not (REPO_DIR / "src/utils/runtime.py").is_file():
    print("main still has the earlier interface; checking out the CLI review branch")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REVIEW_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)

print("Source:", REPO_DIR)
print("Weights:", MODEL_DIR)
print("Results:", OUTPUT_DIR)


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from src.utils.runtime import validate_runtime
validate_runtime()

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
!pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3+cu128torch2.11-cp313-cp313-linux_x86_64.whl
!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg8
!apt-get update
!apt-get install -y  -qq ffmpeg


import torchvision
from torchvision.transforms import InterpolationMode
print("Torchvision:", torchvision.__version__, "FFmpeg:", shutil.which("ffmpeg"))
subprocess.run([sys.executable, str(REPO_DIR / "inference_cli.py"), "--help"], check=True)


## Quick sample

This short sample downloads the default DiT and VAE weights to Drive on first use, then processes up to five frames. Keep the resulting model folder for later sessions.


In [ ]:
sample = REPO_DIR / "examples/inputs/Eyes_212x120.mp4"
sample_output = OUTPUT_DIR / "sample_upscaled.mp4"
subprocess.run([
    sys.executable, str(REPO_DIR / "inference_cli.py"), str(sample),
    "--output", str(sample_output),
    "--model_dir", str(MODEL_DIR),
    "--resolution", "240", "--batch_size", "5", "--load_cap", "5",
    "--video_backend", "ffmpeg",
], check=True)
print("Saved:", sample_output)


## Your video

Upload video.mp4 to the input folder on Drive or change the path. The output is saved in the output folder. For long videos, lower resolution or batch size if the GPU runs out of memory; use VAE tiling for large frames.


In [ ]:
input_video = INPUT_DIR / "video.mp4"
if not input_video.is_file():
    raise FileNotFoundError(f"Upload a video to {input_video}, or edit input_video in this cell")

result_video = OUTPUT_DIR / f"{input_video.stem}_upscaled.mp4"
subprocess.run([
    sys.executable, str(REPO_DIR / "inference_cli.py"), str(input_video),
    "--output", str(result_video),
    "--model_dir", str(MODEL_DIR),
    "--resolution", "1080", "--batch_size", "33",
    "--chunk_size", "330", "--temporal_overlap", "3",
    "--video_backend", "ffmpeg",
], check=True)
print("Saved:", result_video)
